# Movie Recommendation System - Model Notebook

This notebook shows the complete preprocessing and modeling pipeline for a content-based movie recommender. It uses TMDB metadata to create movie tags, vectorizes those tags, computes cosine similarity, and saves the model data as pickle files.

## 1. Import Libraries

We import the libraries needed for data cleaning, feature engineering, vectorization, and saving files.

In [ ]:
import ast
import pickle
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 2. Load the Dataset

Load the TMDB movie and credits files into Pandas DataFrames. Place the files in the `data/` folder before running.

In [ ]:
movies_df = pd.read_csv('data/tmdb_5000_movies.csv')
credits_df = pd.read_csv('data/tmdb_5000_credits.csv')

print('Movies shape:', movies_df.shape)
print('Credits shape:', credits_df.shape)
movies_df.head()

## 3. Merge Movie and Credits Data

We merge the movie and credit information by the movie title so that each row contains cast, crew, and keyword data.

In [ ]:
movies_df = movies_df.merge(credits_df, on='title')
movies_df = movies_df[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew', 'poster_path']]
movies_df.head()

## 4. Data Cleaning

Remove missing values and keep only the columns required for recommendation.

In [ ]:
movies_df.dropna(subset=['genres', 'keywords', 'cast', 'crew'], inplace=True)
movies_df.reset_index(drop=True, inplace=True)
movies_df.shape

## 5. Helper Functions for Feature Extraction

Define functions that will extract useful text from the genre, cast, keyword, and crew JSON-like columns.

In [ ]:
def safe_ast_eval(text):
    try:
        return ast.literal_eval(text)
    except (ValueError, SyntaxError):
        return []

def extract_names(obj_list):
    names = [item['name'].replace(' ', '') for item in obj_list if 'name' in item]
    return names

def get_genres(text):
    return extract_names(safe_ast_eval(text))

def get_keywords(text):
    return extract_names(safe_ast_eval(text))

def get_cast(text):
    cast_list = safe_ast_eval(text)
    return [member['name'].replace(' ', '') for member in cast_list[:5] if 'name' in member]

def get_director(text):
    crew_list = safe_ast_eval(text)
    for member in crew_list:
        if member.get('job') == 'Director':
            return [member['name'].replace(' ', '')]
    return []

## 6. Create Tags

Generate a combined text field called `tags` by merging genres, cast, keywords, and director names.

In [ ]:
movies_df['genres'] = movies_df['genres'].apply(get_genres)
movies_df['keywords'] = movies_df['keywords'].apply(get_keywords)
movies_df['cast'] = movies_df['cast'].apply(get_cast)
movies_df['director'] = movies_df['crew'].apply(get_director)
movies_df['overview'] = movies_df['overview'].fillna('')

movies_df['tags'] = movies_df['genres'] + movies_df['keywords'] + movies_df['cast'] + movies_df['director']
movies_df['tags'] = movies_df['tags'].apply(lambda x: ' '.join(x).lower())

movies_df[['title', 'tags']].head()

## 7. Vectorize Tags

Use `CountVectorizer` to turn text tags into numeric vectors that can be compared with cosine similarity.

In [ ]:
vectorizer = CountVectorizer(max_features=5000, stop_words='english')
tags_matrix = vectorizer.fit_transform(movies_df['tags']).toarray()
tags_matrix.shape

## 8. Compute Cosine Similarity

Generate a similarity matrix from the vectorized tags. Each movie can now be compared to each other movie.

In [ ]:
similarity = cosine_similarity(tags_matrix)
similarity.shape

## 9. Save the Model Data

Save the preprocessed movies and similarity matrix as pickle files for the Streamlit app.

In [ ]:
with open('movies.pkl', 'wb') as file:
    pickle.dump(movies_df[['movie_id', 'title', 'overview', 'poster_path', 'tags']], file)

with open('similarity.pkl', 'wb') as file:
    pickle.dump(similarity, file)

print('Saved movies.pkl and similarity.pkl')

## 10. Recommendation Function (Optional)

A simple function to test recommendations inside the notebook.

In [ ]:
def recommend(movie_name, movies, similarity_matrix):
    if movie_name not in movies['title'].values:
        return []

    idx = movies[movies['title'] == movie_name].index[0]
    distances = list(enumerate(similarity_matrix[idx]))
    distances = sorted(distances, key=lambda x: x[1], reverse=True)

    recommended_movies = []
    for i in distances[1:6]:
        recommended_movies.append(movies.iloc[i[0]]['title'])
    return recommended_movies

recommend('The Dark Knight', movies_df, similarity)